# Week 1 – Werkcollege 2: Testen, toernooi-overzicht en peer review

Vandaag lever je niets nieuws in. Dit werkcollege heeft vier onderdelen: live testen, hoe een API werkt, een echt toernooi tussen alle bots van de klas met pandas verkennen, en peer review.

## Tijdsindicatie

| Moment | Duur | Onderdeel |
|---|---|---|
| Live testen | 15 min | Bot tegen klasgenoot testen |
| Hoe werkt een API | 15 min | Requests, methodes, status codes |
| Toernooi-overzicht met pandas | 20 min | Echt toernooi ophalen, overzicht per bot bouwen |
| Peer review via de API | 25 min | 3 grafieken ophalen en beoordelen |
| Aftrap groepscase | 15 min | Uitleg door docent |


## Deel 1 — Live testen (15 min)

Zoek een klasgenoot op. Wissel je `mijn_bot_week1.py` bestand uit (of gebruik een gedeelde map/repo). Laat beide bots dezelfde set testhanden zien en vergelijk de uitkomsten.

Dit is geen inleveropdracht — het is bedoeld om te zien of jullie bots verschillend reageren, en om bugs te vinden die je zelf niet zag.


In [ ]:
import sys
sys.path.append('.')

# Importeer je eigen bot en die van je klasgenoot.
# Pas de bestandsnamen aan naar wat jullie hebben.
from mijn_bot_week1 import kies_actie as bot_jij
# from bot_klasgenoot import kies_actie as bot_klasgenoot

test_handen = [["A", "A"], ["K", "Q"], ["7", "2"], ["A", "5"], ["J", "J"]]

for hand in test_handen:
    print(hand, "->", bot_jij(hand))
    # print(hand, "->", bot_klasgenoot(hand))


## Deel 2 — Hoe werkt een API (15 min)

Je hebt vorige week al met `lever_in()` iets naar de API gestuurd, zonder dat we precies uitlegden wat daarbij gebeurt. Dat halen we nu in, want je gebruikt het patroon zo dadelijk opnieuw om een heel toernooi op te halen.

Introductie in API's: https://datalab01.ict.hva.nl/w/hdGvm2AW9HbCiU4Lb4uJr2

### Client en server

Een API (Application Programming Interface) is een manier voor twee programma's om met elkaar te praten. Jouw notebook is de **client**: die stuurt een **request**. Ergens anders draait de **server** (in dit geval `main.py`, met FastAPI): die verwerkt de request en stuurt een **response** terug.

Een request bestaat altijd uit een aantal vaste onderdelen:

- **methode** — wat je wilt doen. `GET` = iets ophalen. `POST` = iets versturen of aanmaken.
- **url** — welk endpoint je aanspreekt, bijvoorbeeld `/status/student1/1`.
- **headers** — extra informatie over de request, zoals je `Authorization`-token (zodat de server weet wie je bent).
- **body** — de data die je meestuurt, bij een `POST`. Bij een `GET` is er meestal geen body.

De response die je terugkrijgt heeft ook een vaste vorm:

- **status code** — een getal dat in één oogopslag zegt of het gelukt is. `200` = gelukt. `401` = niet geautoriseerd (verkeerd token). `404` = bestaat niet. `500` = server-fout.
- **body** — meestal JSON, met de eigenlijke inhoud van het antwoord.

### Stap voor stap: een GET-request

We proberen dit uit met het `/status`-endpoint, dat je al kent uit vorige week (het zegt of je inzending goedgekeurd is).

**Stap 1: zonder token.** Dit moet mislukken — dat is precies de bedoeling, zo zie je een `401`.


In [ ]:
import requests

API_URL = "http://localhost:8000"
STUDENT_ID = "vul_hier_je_student_id_in"
TOKEN = "vul_hier_je_token_in"

response_zonder_token = requests.get(f"{API_URL}/status/{STUDENT_ID}/1")
print("status code:", response_zonder_token.status_code)
print("body:", response_zonder_token.json())


**Stap 2: met token.** Nu voeg je de `Authorization`-header toe. Vergelijk de status code met hierboven.


In [ ]:
response = requests.get(
    f"{API_URL}/status/{STUDENT_ID}/1",
    headers={"Authorization": f"Bearer {TOKEN}"},
)
print("status code:", response.status_code)
print("body:", response.json())


**Stap 3: `.status_code` en `.json()` apart bekijken.** `response` is een Python-object met daarin de hele response. Je gebruikt steeds twee onderdelen ervan:

- `response.status_code` — het getal, om te checken of het gelukt is
- `response.json()` — de body, omgezet van JSON-tekst naar een Python dict

🤔 Wat zou er gebeuren als je `.json()` aanroept op een response met status code 500 (een servercrash)? Zou daar geldige JSON in staan?

### Overzicht status codes die je deze cursus tegenkomt

| Code | Betekenis | Wanneer je 'm ziet |
|---|---|---|
| 200 | OK | Alles gelukt |
| 400 | Bad request | Je stuurt iets dat niet klopt (bv. dubbele review) |
| 401 | Unauthorized | Verkeerd of ontbrekend token |
| 404 | Not found | Endpoint of anon_id bestaat niet |
| 500 | Server error | Fout in de API zelf, niet in jouw code |

Je gaat dit patroon (methode + url + headers + eventueel body, dan status code + json body checken) hierna meteen opnieuw gebruiken, en de rest van het blok nog vaker.


## Deel 3 — Toernooi-overzicht met pandas (20 min)

Alle technisch goedgekeurde bots van de klas hebben inmiddels echt tegen elkaar gespeeld: de API heeft ze met `PyPokerEngine` verdeeld over meerdere tafels en meerdere simulaties laten spelen, en logt van elke bot elke hand: welke actie hij koos en wat zijn stack daarna was. Dat haal je nu op — met precies het GET-patroon van Deel 2 — en verken je met `pandas`.

Introductie in pandas: https://datalab01.ict.hva.nl/w/d9zd1VbMKK4oJPP1RzM3Sd

### Het toernooi ophalen


In [ ]:
response = requests.get(
    f"{API_URL}/toernooi/1",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
toernooi_resultaat = response.json()

print("aantal deelnemende bots:", toernooi_resultaat["n_bots"])
print("deelnemers:", toernooi_resultaat["namen_deelnemers"])


De eerste keer dat iemand in de klas dit ophaalt, draait de API het toernooi en bewaart de uitslag. Iedereen daarna krijgt exact dezelfde uitslag terug — er wordt niet elke keer opnieuw gesimuleerd.

Let op: alleen bots die bij `/submit` technisch goedgekeurd zijn, spelen mee. Stond jouw bot er niet bij? Check je eigen `/status` van hierboven.

### Van JSON naar DataFrame

`hand_log` is een lijst met dicts — één rij per hand per bot. Dat zet je direct om in een DataFrame, net zoals je bij `pd.read_csv` een tabel krijgt.


In [ ]:
import pandas as pd

toernooi = pd.DataFrame(toernooi_resultaat["hand_log"])
toernooi.head()


In [ ]:
# Wat voor datatypes zitten erin? Hoeveel rijen zijn er?
toernooi.info()


Achtergrond over data exploratie: https://datalab01.ict.hva.nl/w/h9HaRfrvXcSVuF6Yi8cfaG

### Overzicht per bot bouwen

Elke bot heeft meerdere tafels en simulaties gespeeld, dus meerdere eindstanden. Je wilt niet duizenden losse rijen, maar één overzicht: wat is de gemiddelde eindstand van elke bot? Daarvoor groepeer je de rijen per bot. Dit heet `groupby` — je gaat er in Week 3 dieper op in, hier gebruiken we 'm alvast praktisch.

Groupby instructies: https://datalab01.ict.hva.nl/w/115h1kbnT5B4eijPd6Gc95


In [ ]:
eindstand_per_bot = pd.Series(toernooi_resultaat["eindstand_per_bot"]).sort_values(ascending=False)
eindstand_per_bot


Beantwoord kort: wat is jouw eigen gemiddelde eindstand? Sta je boven of onder het klasgemiddelde, en tegen wie zou je willen zien hoe die het deed op de individuele tafels (in `toernooi`, niet in het samengevatte overzicht)?


In [ ]:
# jouw antwoord (als commentaar of met code)



Je gebruikt `eindstand_per_bot` en `toernooi` straks in Werkcollege 3 (Visual Maandag) als basis voor je grafiek — dezelfde `GET /toernooi/1` haalt daar (uit cache) opnieuw exact dezelfde data op.


## Deel 4 — Peer review via de API (25 min)

De Streamlit Hub doet dit automatisch zodra hij live staat: hij haalt 3 anonieme grafieken op en stuurt je beoordeling terug. Hier doe je het met de hand, met dezelfde requests die de Hub straks ook gebruikt.

### Stap 1: grafieken ophalen

`GET /gallery/{week}` geeft een steekproef van 3 anonieme grafieken terug, nooit je eigen inzending.


In [ ]:
response = requests.get(
    f"{API_URL}/gallery/1",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
gallery = response.json()
gallery


### Stap 2: een grafiek bekijken

Als de grafiek met matplotlib is gemaakt, staat 'ie als base64-PNG in `figuur_json`. Zo toon je 'm in het notebook:


In [ ]:
import base64
from IPython.display import Image, display

eerste = gallery[0]
print(eerste["chart"]["titel"])

if eerste["chart"]["library"] == "matplotlib":
    png_bytes = base64.b64decode(eerste["chart"]["figuur_json"])
    display(Image(png_bytes))


### Stap 3: beoordelen op de 3 criteria

Beoordeel elke grafiek op:

- **Focal point** — is in één oogopslag duidelijk waar je naar moet kijken?
- **Kleur & contrast** — is kleur functioneel gebruikt, of is het een papegaaiengrafiek?
- **Actietitel** — vertelt de titel het inzicht, of alleen de variabelen?

Elk criterium een score van 1 (nee, totaal niet) tot 5 (ja, heel duidelijk).

### Stap 4: beoordeling versturen

Dit is een `POST`, want je maakt nu iets nieuws aan (een review) in plaats van iets op te halen. Daarom gaat de data mee in de `json=`-body, niet in `params=`.


In [ ]:
review = {
    "week": 1,
    "anon_id": eerste["anon_id"],
    "focal_point_score": 4,       # pas aan
    "kleur_contrast_score": 3,    # pas aan
    "actietitel_score": 5,        # pas aan
    "opmerking": "optioneel commentaar",
}

response = requests.post(
    f"{API_URL}/peer-review/{STUDENT_ID}",
    json=review,
    headers={"Authorization": f"Bearer {TOKEN}"},
)
print("status code:", response.status_code)
response.json()


Herhaal stap 2 t/m 4 voor de overige 2 grafieken uit `gallery`. Check daarna je status — je hebt nu 3 reviews gegeven, dus `voldaan` zou `true` moeten worden zodra je eigen inzending ook technisch goedgekeurd is.


In [ ]:
response = requests.get(
    f"{API_URL}/status/{STUDENT_ID}/1",
    headers={"Authorization": f"Bearer {TOKEN}"},
)
response.json()


---

## Reflectievragen

🤔 Wat viel je op aan de grafieken van klasgenoten die je beoordeelde?

🎨 Kon je bij elke grafiek in één oogopslag zien waar je naar moest kijken?

🎨 Welke grafiek vond je het duidelijkst, en wat maakte 'm duidelijk?

🤔 Was er een grafiek waarbij de titel niets vertelde over de inhoud? Wat had een betere titel kunnen zijn?

🤔 Waarom stuurt `/peer-review` de data in de body (`json=`) en `/gallery` in de query (`params=`)? Wat zou er misgaan als dat andersom was?

💡 Je bot speelde net tegen willekeurige klasgenoten, op willekeurige tafels. Wat zou er gebeuren met je eindstand als je bot alleen tegen kopieën van zichzelf had gespeeld?

💡 Wat ga je in Week 2 (Visual Maandag) zelf anders aanpakken aan je eigen grafiek?

---

## Deel 5 — Aftrap groepscase (15 min)

De docent introduceert de eerste groepscase. Vanaf donderdag ligt de focus op de case; de poker-opdracht komt terug in Week 3.
